# Forest Cover Analysis: Western Area Peninsula National Park, Sierra Leone

**Module:** EGM722 – Programming for GIS and Remote Sensing  
**Author:** Charlotte Bond  
**Date:** 2026

---

## Overview

This notebook analyses forest cover change within the **Western Area Peninsula National Park (WAPNP)**,
Sierra Leone, using the Hansen Global Forest Change dataset (GFC v1.11, 2023).

The analysis:

1. Loads the official park boundary from the World Database on Protected Areas (WDPA site ID 19249).
2. Extracts the relevant portion of the Hansen treecover2000 and lossyear rasters using windowed reading.
3. Calculates forest cover statistics and annual forest loss within the park (2001–2023).
4. Computes zonal statistics using `rasterstats`.
5. Produces a static map with `cartopy` and an interactive map with `folium`.


In [ ]:
# Standard library
import os
import json
import warnings

# Data handling
import numpy as np
import pandas as pd

# Geospatial – vector
import geopandas as gpd

# Geospatial – raster
import rasterio
import rasterio.windows
import rasterio.mask
from rasterio.plot import show as rasterio_show

# Zonal statistics
from rasterstats import zonal_stats

# Mapping – static
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Mapping – interactive
import folium

# Suppress minor warnings for cleaner output
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

print("All packages imported successfully.")


## 1. Configuration

All paths, URLs, and analysis parameters are defined here so they can be changed
in one place without editing the rest of the notebook.


In [ ]:
# Directory paths 
DATA_DIR = "data"
OUTPUT_DIR = "outputs"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Input data ────────────────────────────────────────────────────────────────
BOUNDARY_FILE = os.path.join(
    DATA_DIR,
    "WDPA_WDOECM_May2026_Public_19249_shp_0",
    "WDPA_WDOECM_May2026_Public_19249_shp-polygons.shp"
)

# Bounding box used to extract a spatial subset of the Hansen tiles.
# Values are in decimal degrees (WGS84). A small buffer is added around
# the park boundary so the maps include some surrounding context.
# Bounding box derived from official WDPA boundary with 0.05 degree buffer
BBOX = {
    'west':  -13.35,
    'south':  7.99,
    'east':  -12.90,
    'north':  8.55
}

# ── Hansen GFC tile URLs (no authentication required) ────────────────────────
# Sierra Leone falls within the 10N_010W tile (0–10°N, 10–20°W).
# The files are Cloud-Optimised GeoTIFFs, so rasterio can read a spatial
# window without downloading the full tile (~150 MB and ~20 MB respectively).
HANSEN_BASE = (
    "https://storage.googleapis.com/earthenginepartners-hansen/"
    "GFC-2023-v1.11"
)
HANSEN_TILE = "10N_020W"

HANSEN_URLS = {
    'treecover2000': f"{HANSEN_BASE}/Hansen_GFC-2023-v1.11_treecover2000_{HANSEN_TILE}.tif",
    'lossyear':      f"{HANSEN_BASE}/Hansen_GFC-2023-v1.11_lossyear_{HANSEN_TILE}.tif",
}

# Local paths where the extracted subsets will be cached
LOCAL_RASTERS = {
    'treecover2000': os.path.join(DATA_DIR, "treecover2000_wapnp.tif"),
    'lossyear':      os.path.join(DATA_DIR, "lossyear_wapnp.tif"),
}

# ── Analysis parameters ───────────────────────────────────────────────────────
# Forest is defined as pixels with >= FOREST_THRESHOLD % canopy cover.
# A 30 % threshold is consistent with the FAO forest definition.
FOREST_THRESHOLD = 30

# Hansen loss year encoding: 1 = 2001, …, 23 = 2023 (for GFC-2023 v1.11)
LOSS_YEAR_OFFSET = 2000   # add to Hansen code to get calendar year
LOSS_YEAR_END    = 2023

# Approximate area of one Hansen pixel at the park's latitude (~8.3°N).
# Hansen GFC data has ~30 m resolution. At 8.3°N, cos(8.3°) ≈ 0.989,
# so pixel width ≈ 29.7 m, height ≈ 30 m → area ≈ 891 m² ≈ 0.089 ha.
PIXEL_AREA_HA = 0.089  # hectares per pixel

# Coordinate reference system for the analysis
CRS_WGS84 = "EPSG:4326"

print("Configuration complete.")
print(f"  Data directory   : {os.path.abspath(DATA_DIR)}")
print(f"  Output directory : {os.path.abspath(OUTPUT_DIR)}")


## 2. Data Loading

### 2.1 Park Boundary

The park boundary is the official WDPA polygon for site ID 19249 (Western Area Peninsula
Forest National Park, Sierra Leone), downloaded from
[Protected Planet](https://www.protectedplanet.net/19249) in May 2026
(UNEP-WCMC and IUCN, 2026). It is stored as a shapefile in
`data/WDPA_WDOECM_May2026_Public_19249_shp_0/` and covers approximately 179 km².

### 2.2 Hansen Global Forest Change Rasters

Forest cover data come from the **Hansen Global Forest Change** dataset
(Hansen et al., 2013), accessed through Google Cloud Storage. Two layers are used:

- **treecover2000** – canopy cover percentage (0–100) for the year 2000.
- **lossyear** – year of first detected forest loss (0 = no loss, 1 = 2001, …, 23 = 2023).

Because these are Cloud-Optimised GeoTIFFs (COGs), `rasterio` can read only the
spatial window corresponding to the park extent, avoiding the need to download
the full tile.


In [ ]:
def load_park_boundary(filepath):
    """
    Load the park boundary from a GeoJSON file and ensure it is in WGS84.

    Parameters
    ----------
    filepath : str
        Path to the GeoJSON file containing the park boundary polygon.

    Returns
    -------
    geopandas.GeoDataFrame
        GeoDataFrame of the park boundary reprojected to EPSG:4326 (WGS84).

    Raises
    ------
    FileNotFoundError
        If ``filepath`` does not exist.
    """
    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"Boundary file not found: {filepath}\n"
            "Ensure data/wapnp_boundary.geojson is present in the repository."
        )

    gdf = gpd.read_file(filepath)

    # Set or reproject to WGS84 so it aligns with the Hansen rasters
    if gdf.crs is None:
        gdf = gdf.set_crs(CRS_WGS84)
    else:
        gdf = gdf.to_crs(CRS_WGS84)

    return gdf


In [ ]:
def extract_raster_subset(url, bbox, output_path, force_redownload=False):
    """
    Read a spatial window of a remote Cloud-Optimised GeoTIFF and save it locally.

    Uses GDAL VSI curl so only the relevant data block is transferred over the
    network rather than the entire file.

    Parameters
    ----------
    url : str
        HTTPS URL of the remote COG file.
    bbox : dict
        Bounding box with keys ``'west'``, ``'south'``, ``'east'``, ``'north'``
        in decimal degrees (WGS84).
    output_path : str
        Local path at which to save the extracted GeoTIFF subset.
    force_redownload : bool, optional
        If ``True``, overwrite an existing local file. Default is ``False``
        (skip the download if the file already exists).

    Returns
    -------
    str
        Path to the saved local raster file.

    Raises
    ------
    RuntimeError
        If the remote file cannot be read (e.g. no internet access, or GDAL
        is built without network support).
    """
    # Skip if file already exists and re-download is not requested
    if os.path.exists(output_path) and not force_redownload:
        print(f"  Already cached: {output_path}")
        return output_path

    print(f"  Downloading subset from remote COG …")
    print(f"    URL  : {url}")
    print(f"    BBOX : {bbox}")

    try:
        with rasterio.open(url) as src:
            # Translate the geographic bounding box into a pixel window
            window = rasterio.windows.from_bounds(
                left=bbox['west'],
                bottom=bbox['south'],
                right=bbox['east'],
                top=bbox['north'],
                transform=src.transform
            )

            # Read only the windowed pixels (band 1 for single-band Hansen layers)
            data = src.read(1, window=window)
            window_transform = src.window_transform(window)

            # Copy and update the raster profile for the output file
            profile = src.profile.copy()
            profile.update({
                'height':    data.shape[0],
                'width':     data.shape[1],
                'transform': window_transform,
                'driver':    'GTiff',
                'compress':  'lzw',
                'count':     1
            })

            with rasterio.open(output_path, 'w', **profile) as dst:
                dst.write(data, 1)

    except Exception as exc:
        raise RuntimeError(
            f"Could not read remote raster: {exc}\n"
            "Check your internet connection. If the error mentions SSL, try "
            "setting the environment variable GDAL_HTTP_UNSAFESSL=YES."
        ) from exc

    print(f"    Saved : {output_path} "
          f"({data.shape[1]} x {data.shape[0]} pixels)")
    return output_path


In [ ]:
# Load park boundary 
print("Loading park boundary …")
park_gdf = load_park_boundary(BOUNDARY_FILE)
print(f"  Features : {len(park_gdf)}")
print(f"  CRS      : {park_gdf.crs}")
print(f"  Bounds   : {park_gdf.total_bounds.round(4)}")

#  Download/cache Hansen raster subsets 
print("\nExtracting Hansen raster subsets …")
for layer_name, url in HANSEN_URLS.items():
    extract_raster_subset(url, BBOX, LOCAL_RASTERS[layer_name])

print("\nData loading complete.")


## 3. Forest Cover Analysis

### 3.1  Forest cover in 2000
Pixels with canopy cover ≥ 30 % are classified as **forest**, consistent with
the FAO definition (Food and Agriculture Organization, 2020).

### 3.2  Annual forest loss (2001–2023)
The Hansen `lossyear` layer records the calendar year in which each pixel
experienced its first detectable canopy loss. Pixels that were not forest in
2000 are excluded from the loss count.

### 3.3  Zonal statistics
`rasterstats` computes summary statistics (mean, median, etc.) of the
treecover2000 layer within the park boundary polygon.


In [ ]:
def calculate_forest_cover(treecover_array, threshold=FOREST_THRESHOLD):
    """
    Classify pixels as forest or non-forest and compute cover statistics.

    Parameters
    ----------
    treecover_array : numpy.ndarray
        2-D array of Hansen treecover2000 values (0–100 % canopy; 255 = nodata).
    threshold : int, optional
        Minimum canopy-cover percentage to classify a pixel as forest.
        Default is 30 (``FOREST_THRESHOLD`` from configuration).

    Returns
    -------
    dict
        A dictionary with the following keys:

        - ``forest_pixels`` : int – number of pixels classified as forest.
        - ``total_pixels``  : int – total number of valid (non-nodata) pixels.
        - ``forest_percent``: float – percentage of the area classified as forest.
        - ``forest_mask``   : numpy.ndarray – boolean array; ``True`` = forest pixel.
    """
    # Build a mask of valid pixels (nodata value in Hansen data is 255)
    valid_mask = treecover_array < 255
    total_pixels = int(np.sum(valid_mask))

    # Apply the forest threshold within valid pixels
    forest_mask = (treecover_array >= threshold) & valid_mask
    forest_pixels = int(np.sum(forest_mask))

    forest_percent = (forest_pixels / total_pixels * 100) if total_pixels > 0 else 0.0

    return {
        'forest_pixels':  forest_pixels,
        'total_pixels':   total_pixels,
        'forest_percent': round(float(forest_percent), 2),
        'forest_mask':    forest_mask
    }


In [ ]:
def calculate_annual_loss(lossyear_array, forest_mask):
    """
    Compute annual forest loss counts from the Hansen lossyear layer.

    Only pixels that were classified as forest in 2000 (``forest_mask == True``)
    are counted as losses. This prevents agricultural or bare-soil pixels being
    incorrectly flagged as forest losses.

    Parameters
    ----------
    lossyear_array : numpy.ndarray
        2-D array of Hansen lossyear values. Encoding: 0 = no loss detected;
        1 = loss in 2001; 2 = loss in 2002; …; 23 = loss in 2023.
    forest_mask : numpy.ndarray
        Boolean array where ``True`` indicates a pixel was forest in 2000.
        Typically the ``'forest_mask'`` value returned by
        :func:`calculate_forest_cover`.

    Returns
    -------
    pandas.DataFrame
        DataFrame indexed by calendar year with columns:

        - ``pixels_lost``      – number of forest pixels lost in that year.
        - ``cumulative_loss``  – running total of pixels lost since 2001.
        - ``hectares_lost``    – area lost per year in hectares.
        - ``cumulative_ha_lost`` – cumulative area lost in hectares.
    """
    # Zero out loss codes that fall outside the forest mask
    loss_in_forest = np.where(forest_mask, lossyear_array, 0)

    # Iterate over every possible year code (1–23 covers 2001–2023)
    records = []
    for code in range(1, LOSS_YEAR_END - LOSS_YEAR_OFFSET + 1):
        year = code + LOSS_YEAR_OFFSET
        pixels_lost = int(np.sum(loss_in_forest == code))
        records.append({'year': year, 'pixels_lost': pixels_lost})

    df = pd.DataFrame(records).set_index('year')
    df['cumulative_loss'] = df['pixels_lost'].cumsum()

    # Convert pixel counts to hectares using the configured pixel area
    df['hectares_lost']      = (df['pixels_lost']   * PIXEL_AREA_HA).round(1)
    df['cumulative_ha_lost'] = (df['cumulative_loss'] * PIXEL_AREA_HA).round(1)

    return df


In [ ]:
def compute_park_zonal_stats(raster_path, park_boundary_gdf,
                              stats_list=None, nodata_value=255):
    """
    Compute zonal statistics of a raster within the park boundary polygon.

    Wraps ``rasterstats.zonal_stats`` to handle CRS reprojection and return
    results as a GeoDataFrame with the statistics added as columns.

    Parameters
    ----------
    raster_path : str
        Path to the local raster file.
    park_boundary_gdf : geopandas.GeoDataFrame
        GeoDataFrame containing the zone polygon(s).
    stats_list : list of str, optional
        Statistics to compute. Defaults to
        ``['min', 'max', 'mean', 'median', 'count']``.
    nodata_value : int or float, optional
        Nodata value in the raster to exclude from calculations. Default is 255
        (the Hansen nodata value).

    Returns
    -------
    geopandas.GeoDataFrame
        Copy of ``park_boundary_gdf`` with one new column per requested
        statistic, prefixed with ``'tc_'`` (for treecover).
    """
    if stats_list is None:
        stats_list = ['min', 'max', 'mean', 'median', 'count']

    # Reproject the boundary to match the raster CRS before passing to rasterstats
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs

    boundary_reprojected = park_boundary_gdf.to_crs(raster_crs)

    # Run zonal statistics
    raw_stats = zonal_stats(
        boundary_reprojected,
        raster_path,
        stats=stats_list,
        nodata=nodata_value
    )

    # Attach results to a copy of the input GeoDataFrame
    result_gdf = park_boundary_gdf.copy()
    for stat in stats_list:
        result_gdf[f'tc_{stat}'] = [row.get(stat) for row in raw_stats]

    return result_gdf


In [ ]:
# Clip rasters to the park boundary 
print("Clipping rasters to park boundary …")

clipped = {}
clip_transforms = {}

for layer, path in LOCAL_RASTERS.items():
    with rasterio.open(path) as src:
        # rasterio.mask.mask requires geometries in the same CRS as the raster
        park_in_raster_crs = park_gdf.to_crs(src.crs)
        geoms = [geom.__geo_interface__ for geom in park_in_raster_crs.geometry]

        clipped_data, clipped_transform = rasterio.mask.mask(
            src, geoms, crop=True, nodata=255
        )
        # mask() returns a 3-D array (bands, rows, cols); squeeze to 2-D
        clipped[layer] = clipped_data[0]
        clip_transforms[layer] = clipped_transform

    print(f"  {layer}: clipped to {clipped[layer].shape[1]} x {clipped[layer].shape[0]} px")

# Forest cover in 2000 nprint("\nCalculating forest cover …")
cover_stats = calculate_forest_cover(clipped['treecover2000'])
print(f"  Forest pixels  : {cover_stats['forest_pixels']:,}")
print(f"  Total pixels   : {cover_stats['total_pixels']:,}")
print(f"  Forest cover   : {cover_stats['forest_percent']} %")

# Annual forest loss 
print("\nCalculating annual loss …")
loss_df = calculate_annual_loss(clipped['lossyear'], cover_stats['forest_mask'])

total_loss = loss_df['pixels_lost'].sum()
loss_pct   = total_loss / cover_stats['forest_pixels'] * 100 if cover_stats['forest_pixels'] > 0 else 0
print(f"  Total pixels lost (2001–2023) : {total_loss:,}")
print(f"  As % of 2000 forest           : {loss_pct:.1f} %")

# Zonal statistics 
print("\nComputing zonal statistics (treecover2000) …")
zonal_gdf = compute_park_zonal_stats(LOCAL_RASTERS['treecover2000'], park_gdf)

print("  Zonal statistics (canopy cover %):")
for col in [c for c in zonal_gdf.columns if c.startswith('tc_')]:
    val = zonal_gdf[col].iloc[0]
    stat_name = col.replace('tc_', '')
    print(f"    {stat_name:8s}: {val:.1f}" if val is not None else f"    {stat_name:8s}: N/A")


In [ ]:
# Print a formatted summary table 
total_ha_lost = loss_df['hectares_lost'].sum()

print("=" * 55)
print("  WAPNP FOREST COVER SUMMARY")
print("=" * 55)
print(f"  Forest cover in 2000 (≥30 % canopy)  : {cover_stats['forest_percent']:>6.1f} %")
print(f"  Total forest loss 2001–2023           : {total_loss:>6,} pixels")
print(f"  Total forest loss 2001–2023           : {total_ha_lost:>6.1f} ha")
print(f"  Loss as % of year-2000 forest         : {loss_pct:>6.1f} %")
print(f"  Mean canopy cover in 2000 (park-wide) : {zonal_gdf['tc_mean'].iloc[0]:>6.1f} %")
print("=" * 55)

# Show the per-year loss table with hectares
print("\nAnnual forest loss:")
print(loss_df[['pixels_lost', 'hectares_lost', 'cumulative_ha_lost']].to_string())


## 4. Visualisation

### 4.1  Static map
A two-panel figure produced with `cartopy` and `matplotlib`:
- **Left** – tree canopy cover in 2000 within the park.
- **Right** – bar chart of annual pixel loss (2001–2023) with cumulative loss overlay.

### 4.2  Interactive map
A `folium` map showing the park boundary with a popup summary of key statistics,
saved to `outputs/wapnp_interactive_map.html`.


In [ ]:
def create_static_map(park_gdf, treecover_clipped, tc_transform,
                      loss_df, output_path=None):
    """
    Create a two-panel static figure: forest cover map and annual loss chart.

    Parameters
    ----------
    park_gdf : geopandas.GeoDataFrame
        Park boundary in WGS84.
    treecover_clipped : numpy.ndarray
        2-D array of treecover2000 values clipped to the park extent.
    tc_transform : affine.Affine
        Affine transform of the clipped treecover raster.
    loss_df : pandas.DataFrame
        Annual loss table (output of :func:`calculate_annual_loss`).
    output_path : str, optional
        If given, the figure is saved to this path at 150 dpi.

    Returns
    -------
    matplotlib.figure.Figure
        The completed figure object.
    """
    fig = plt.figure(figsize=(14, 6))

    # ── Left panel: canopy cover map ──────────────────────────────────────────
    crs_pc = ccrs.PlateCarree()
    ax_map = fig.add_subplot(1, 2, 1, projection=crs_pc)
    ax_map.set_title(
        "Tree Canopy Cover in 2000\n(Western Area Peninsula National Park)",
        fontsize=11, pad=10
    )

    # Compute the geographic extent of the clipped raster
    h, w = treecover_clipped.shape
    left   = tc_transform.c
    top    = tc_transform.f
    right  = left + w * tc_transform.a
    bottom = top  + h * tc_transform.e
    raster_extent = [left, right, bottom, top]

    # Display canopy cover (mask nodata = 255 as NaN)
    display_data = np.where(treecover_clipped < 255,
                            treecover_clipped / 100.0, np.nan)
    img = ax_map.imshow(
        display_data, extent=raster_extent, transform=crs_pc,
        cmap='YlGn', vmin=0, vmax=1, alpha=0.85, origin='upper'
    )
    plt.colorbar(img, ax=ax_map, shrink=0.6, label='Canopy cover fraction')

    # Overlay the park boundary
    park_gdf.boundary.plot(ax=ax_map, edgecolor='red', linewidth=1.8,
                           transform=crs_pc, zorder=5)

    # Add standard cartopy features
    ax_map.add_feature(cfeature.COASTLINE, linewidth=0.7, zorder=4)
    ax_map.add_feature(cfeature.OCEAN, facecolor='#d0eaf8', alpha=0.6, zorder=3)

    bounds = park_gdf.total_bounds  # [minx, miny, maxx, maxy]
    ax_map.set_extent(
        [bounds[0] - 0.06, bounds[2] + 0.06, bounds[1] - 0.06, bounds[3] + 0.06],
        crs=crs_pc
    )

    gl = ax_map.gridlines(draw_labels=True, linestyle='--', alpha=0.4, linewidth=0.5)
    gl.top_labels   = False
    gl.right_labels = False

    boundary_line = plt.Line2D([0], [0], color='red', linewidth=1.8,
                                label='Park boundary')
    ax_map.legend(handles=[boundary_line], loc='lower left', fontsize=8)

    # ── Right panel: annual loss bar chart ───────────────────────────────────
    ax_bar = fig.add_subplot(1, 2, 2)
    ax_bar.set_title(
        "Annual Forest Loss 2001–2023\n(within park boundary)",
        fontsize=11, pad=10
    )

    ax_bar.bar(loss_df.index, loss_df['hectares_lost'],
               color='#8B0000', alpha=0.75, label='Annual loss (ha)')

    # Cumulative loss on a secondary y-axis
    ax_cum = ax_bar.twinx()
    ax_cum.plot(loss_df.index, loss_df['cumulative_ha_lost'],
                color='#FF8C00', linewidth=2, marker='o', markersize=4,
                label='Cumulative loss (ha)')

    ax_bar.set_xlabel("Year", fontsize=10)
    ax_bar.set_ylabel("Hectares lost (per year)", fontsize=10, color='#8B0000')
    ax_cum.set_ylabel("Cumulative hectares lost", fontsize=10, color='#FF8C00')
    ax_bar.tick_params(axis='x', rotation=45)

    # Combine legends from both axes
    lines1, labels1 = ax_bar.get_legend_handles_labels()
    lines2, labels2 = ax_cum.get_legend_handles_labels()
    ax_bar.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=8)
    ax_bar.grid(axis='y', alpha=0.3)

    plt.tight_layout()

    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved to: {output_path}")

    return fig


In [ ]:
def create_interactive_map(park_gdf, cover_stats, loss_df):
    """
    Build an interactive Folium map centred on the park.

    The park boundary is rendered as a clickable GeoJSON layer with a popup
    showing forest cover statistics. A CartoDB Positron basemap is used as the
    default tile layer.

    Parameters
    ----------
    park_gdf : geopandas.GeoDataFrame
        Park boundary in WGS84.
    cover_stats : dict
        Forest cover statistics (output of :func:`calculate_forest_cover`).
    loss_df : pandas.DataFrame
        Annual loss statistics (output of :func:`calculate_annual_loss`).

    Returns
    -------
    folium.Map
        Interactive map object. Display with the variable name in a Jupyter
        cell, or save with ``.save('path/to/file.html')``.
    """
    bounds  = park_gdf.total_bounds
    centre  = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]

    total_loss    = int(loss_df['pixels_lost'].sum())
    total_loss_ha = round(loss_df['hectares_lost'].sum(), 1)
    loss_pct      = (total_loss / cover_stats['forest_pixels'] * 100
                     if cover_stats['forest_pixels'] > 0 else 0)

    # ── Base map ──────────────────────────────────────────────────────────────
    m = folium.Map(location=centre, zoom_start=11, tiles='CartoDB positron')

    # ── Park boundary layer ───────────────────────────────────────────────────
    popup_html = (
        "<div style='font-family:Arial,sans-serif; width:240px;'>"
        "<h4 style='color:#2d6a4f; margin-bottom:6px;'>"
        "Western Area Peninsula National Park</h4>"
        "<table style='width:100%; font-size:12px;'>"
        "<tr><td><b>Country</b></td><td>Sierra Leone</td></tr>"
        "<tr><td><b>Forest cover (2000)</b></td>"
        "<td>{fc:.1f} %</td></tr>"
        "<tr><td><b>Total loss 2001-2023</b></td>"
        "<td>{tl:,} pixels ({tlha:.1f} ha)</td></tr>"
        "<tr><td><b>Loss as % of 2000 forest</b></td>"
        "<td>{lp:.1f} %</td></tr>"
        "</table>"
        "<hr style='margin:6px 0;'/>"
        "<small style='color:#666;'>Source: Hansen GFC v1.11 (2023)</small>"
        "</div>"
    ).format(
        fc=cover_stats['forest_percent'],
        tl=total_loss,
        tlha=total_loss_ha,
        lp=loss_pct
    )

    folium.GeoJson(
        park_gdf.__geo_interface__,
        name="Park boundary",
        style_function=lambda _: {
            'fillColor':   '#2d6a4f',
            'color':       '#d62828',
            'weight':       2.5,
            'fillOpacity':  0.25
        },
        tooltip=folium.Tooltip("Western Area Peninsula NP – click for details"),
        popup=folium.Popup(popup_html, max_width=270)
    ).add_to(m)

    folium.LayerControl().add_to(m)

    return m


In [ ]:
# Static map 
print("Creating static map …")
static_map_path = os.path.join(OUTPUT_DIR, "wapnp_forest_cover_map.png")

fig = create_static_map(
    park_gdf,
    clipped['treecover2000'],
    clip_transforms['treecover2000'],
    loss_df,
    output_path=static_map_path
)
plt.show()

# Interactive Folium map 
print("\nCreating interactive map …")
interactive_map = create_interactive_map(park_gdf, cover_stats, loss_df)

interactive_map_path = os.path.join(OUTPUT_DIR, "wapnp_interactive_map.html")
interactive_map.save(interactive_map_path)
print(f"Interactive map saved to: {interactive_map_path}")

# Display the map inline in the notebook
interactive_map


## 5. Discussion

The analysis quantifies forest cover and annual forest loss within the Western Area
Peninsula National Park between 2001 and 2023 using the Hansen Global Forest Change
dataset (Hansen et al., 2013).

In 2000, 98.7% of valid pixels within the park exceeded the 30% canopy-cover threshold,
with a mean canopy cover of 63.6% and a median of 62.0%. This high baseline is consistent
with the park's character as a relatively intact fragment of Upper Guinea forest at the
start of the study period (Norman et al., 2018).

Between 2001 and 2023, 22,221 pixels (1,977.7 ha) were recorded as having lost forest
cover, representing 9.6% of the year-2000 forest extent. Annual losses were low from
2001 to 2013, averaging approximately 11.8 ha per year and peaking at 33.7 ha in 2013.
From 2014 the rate increased sharply: 61.3 ha in 2014, 85.2 ha in 2015, and 292.1 ha
in 2016. Losses remained elevated through the later years of the record, with 276.4 ha
in 2020 and 299.1 ha in 2021, the highest annual figure in the dataset. Cumulative
loss reached 1,977.7 ha by the end of 2023.

The cause of the acceleration from 2014 cannot be established from the Hansen data alone.
The lossyear layer records the year of first detectable canopy loss per pixel but cannot
distinguish between permanent clearance and temporary disturbance, nor identify specific
drivers such as agricultural encroachment, logging, or fire. The 30% canopy-cover
threshold used to define forest is consistent with the FAO operational definition
(FAO, 2020) but is to some degree arbitrary; a stricter threshold would reduce the
baseline cover figure, though at this park almost all pixels exceed 30%, so the effect
on loss totals would be minor. Cloud cover in the wet season can also delay loss
detection by one or more years, meaning some values in the time series may reflect
detection lag rather than the true date of disturbance.

---

### References

Food and Agriculture Organization of the United Nations (FAO) (2020)
*Global Forest Resources Assessment 2020: Main Report*. Rome: FAO.
doi: [10.4060/ca9825en](https://doi.org/10.4060/ca9825en)

Hansen, M.C. et al. (2013) High-resolution global maps of 21st-century
forest cover change. *Science*, 342(6160), pp. 850–853.
doi: [10.1126/science.1244693](https://doi.org/10.1126/science.1244693)

Kluyver, T. et al. (2016) Jupyter Notebooks – a publishing format for
reproducible computational workflows, in Loizides, F. and Scmidt, B. (eds).
*20th International Conference on Electronic Publishing (01/01/16)*.
IOS Press, pp. 87–90. doi: [10.3233/978-1-61499-649-1-87](https://doi.org/10.3233/978-1-61499-649-1-87)

Norman, M. et al. (2018) Threatened birds of the Western Area Peninsula, Sierra Leone.
*Malimbus*, 40(1), pp. 1–22.

UNEP-WCMC and IUCN (2026) *Protected Planet: The World Database on Protected Areas
(WDPA)*. Cambridge: UNEP-WCMC and IUCN. Available at: www.protectedplanet.net
(Accessed: May 2026).
